In [1]:
import pandas as pd
import numpy as np
from scipy.stats import kurtosis, entropy
import re  # For robust extraction of numeric values
from scipy.fftpack import fft
import scipy.signal as signal
import statsmodels.api as sm
from itertools import combinations
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression


# List of file paths
file_paths = [f"part {i}.csv" for i in range(1, 11)]

# Select only the three vibration signal channels
channels = ['Channel1 [g]', 'Channel2 [g]', 'Channel3 [g]']

# Define windowing parameters
window_size = 1024  # Window length for FFT
stft_window_size = 512  # Window length for STFT
overlap = 0.5  # 50% overlap
step = int(stft_window_size * (1 - overlap))  # Step size
sampling_rate = 1000  # Adjust based on actual data

# Y_true values
Y_true = [2.763, 2.45, 2.394, 2.48, 2.334, 2.327, 2.622, 2.556, 2.344, 2.591]  # Replace with actual values

# Function to compute time-domain features
def calculate_time_features(vibration_signal):
    peak = np.max(vibration_signal)  # Peak value
    rms = np.sqrt(np.mean(vibration_signal**2))  # RMS
    crest_factor = peak / rms  # Crest Factor
    kurt = kurtosis(vibration_signal)  # Kurtosis
    return rms, peak, crest_factor, kurt

# Function to compute frequency-domain features with windowing
def calculate_frequency_features(signal_data):
    num_windows = (len(signal_data) - window_size) // window_size + 1
    psd_vals, dominant_freq_vals, total_energy_vals, spectral_entropy_vals = [], [], [], []
    
    for i in range(num_windows):
        start = i * window_size
        end = start + window_size
        if end > len(signal_data):
            break
        segment = signal_data[start:end]
        
        fft_transformed = np.abs(fft(segment))
        freqs = np.fft.fftfreq(len(segment), d=1/sampling_rate)
        psd_vals.append(np.mean(fft_transformed**2))
        dominant_freq_vals.append(freqs[np.argmax(fft_transformed)])
        total_energy_vals.append(np.sum(fft_transformed**2))
        psd_norm = fft_transformed / np.sum(fft_transformed)
        spectral_entropy_vals.append(entropy(np.abs(psd_norm)))
    
    return np.mean(psd_vals), np.mean(dominant_freq_vals), np.mean(total_energy_vals), np.mean(spectral_entropy_vals)

# Function to compute the STFT features
def compute_stft_features(signal_data, fs, window_size):
    frequencies, times, Zxx = signal.stft(signal_data, fs=fs, nperseg=window_size)
    magnitude = np.abs(Zxx) + 1e-10  # Small value to prevent division errors
    spectral_centroid = np.sum(frequencies[:, np.newaxis] * magnitude, axis=0) / np.sum(magnitude, axis=0)
    mean_centroid = np.nanmean(spectral_centroid)  # Handle NaNs
    spectral_bandwidth = np.sqrt(np.sum(((frequencies[:, np.newaxis] - spectral_centroid) ** 2) * magnitude, axis=0) / np.sum(magnitude, axis=0))
    mean_bandwidth = np.nanmean(spectral_bandwidth)  # Handle NaNs
    spectral_entropy = entropy(magnitude, axis=0)
    mean_entropy = np.nanmean(spectral_entropy)  # Handle NaNs
    spectral_kurtosis = kurtosis(magnitude, axis=0, fisher=False)
    mean_kurtosis = np.nanmean(spectral_kurtosis)  # Handle NaNs
    return mean_centroid, mean_bandwidth, mean_entropy, mean_kurtosis

# Initialize lists to store results
all_results = []

# Process each CSV file
for idx, file_path in enumerate(file_paths):
    try:
        df = pd.read_csv(file_path)
        match = re.search(r'(\d+)', file_path)
        part_number = int(match.group(1)) if match else 0
        part_no = f"PART {part_number}"
        row_data = [part_no, part_number]
        
        for channel in channels:
            signal_data = df[channel].values
            rms, peak, crest_factor, kurt = calculate_time_features(signal_data)
            psd, dominant_freq, total_energy, spectral_entropy = calculate_frequency_features(signal_data)
            centroid, bandwidth, entropy_val, kurt_val = compute_stft_features(signal_data, sampling_rate, stft_window_size)
            row_data.extend([rms, peak, crest_factor, kurt, psd, dominant_freq, total_energy, spectral_entropy, centroid, bandwidth, entropy_val, kurt_val])
        
        row_data.append(Y_true[idx])  # Add Y_true value corresponding to each part
        all_results.append(row_data)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

# Define column headers
column_headers = ["PART NO.", "PART NO. NUM"]
for ch in range(1, 4):
    column_headers.extend([
        f"Channel {ch} RMS", f"Channel {ch} PEAK", f"Channel {ch} CREST FACTOR", f"Channel {ch} KURTOSIS",
        f"Channel {ch} PSD", f"Channel {ch} Dominant Frequency", f"Channel {ch} Total Energy", f"Channel {ch} Spectral Entropy",
        f"Channel {ch} Spectral Centroid", f"Channel {ch} Spectral Bandwidth", f"Channel {ch} Spectral Entropy (TF)", f"Channel {ch} Spectral Kurtosis"
    ])
column_headers.append("Y_true")  # Add Y_true column

# Create DataFrame and sort it
features_df = pd.DataFrame(all_results, columns=column_headers)
features_df.sort_values(by="PART NO. NUM", inplace=True, ascending=True)
features_df.drop(columns=["PART NO. NUM"], inplace=True)
features_df.reset_index(drop=True, inplace=True)

# Compute R-squared values
X = features_df.drop(columns=["PART NO.", "Y_true"])
Y = features_df["Y_true"]
r2_values = [sm.OLS(Y, sm.add_constant(X[[col]])).fit().rsquared for col in X.columns]
r2_row = pd.DataFrame([{**{col: r2 for col, r2 in zip(X.columns, r2_values)}, 'PART NO.': "R-squared Values", 'Y_true': None}])
features_df = pd.concat([features_df, r2_row], ignore_index=True)


C:\Users\91834\AppData\Local\Temp\ipykernel_30944\4005940846.py:118: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  features_df = pd.concat([features_df, r2_row], ignore_index=True)


In [2]:
# Select top R-squared features within each domain
selected_features = []
feature_groups = ["RMS", "PEAK", "CREST FACTOR", "KURTOSIS", "PSD", "Dominant Frequency", "Total Energy", "Spectral Entropy", "Spectral Centroid", "Spectral Bandwidth", "Spectral Entropy (TF)", "Spectral Kurtosis"]

for feature in feature_groups:
    relevant_cols = [col for col in X.columns if feature in col]
    if relevant_cols:
        top_feature = max(relevant_cols, key=lambda col: sm.OLS(Y, sm.add_constant(X[[col]])).fit().rsquared)
        selected_features.append(top_feature)
# Define output file path
output_file_path = "Vibration_Analysis_Combined.xlsx"

# Create a DataFrame with top selected features
selected_features_df = features_df[["PART NO."] + selected_features + ["Y_true"]]
with pd.ExcelWriter(output_file_path, engine="xlsxwriter") as writer:
    features_df.to_excel(writer, sheet_name="Combined_Results", index=False)
    selected_features_df.to_excel(writer, sheet_name="Top_Features", index=False)

print(f"Results saved in {output_file_path} with all feature combinations listed in an additional sheet.")

Results saved in Vibration_Analysis_Combined.xlsx with all feature combinations listed in an additional sheet.


In [3]:
# Generate all possible combinations of 4 features
all_combinations = list(combinations(selected_features, 4))

In [10]:
# Load the Excel file
file_path = "Vibration_Analysis_Combined.xlsx"
xls = pd.ExcelFile(file_path)

# Load the 'Top_Features' sheet
top_features_df = xls.parse('Top_Features')

# Define the top 12 features
top_12_features = [
    'Channel 2 RMS', 'Channel 3 PEAK', 'Channel 1 CREST FACTOR',
    'Channel 3 KURTOSIS', 'Channel 2 PSD', 'Channel 2 Dominant Frequency',
    'Channel 2 Total Energy', 'Channel 3 Spectral Entropy',
    'Channel 2 Spectral Centroid', 'Channel 3 Spectral Bandwidth',
    'Channel 2 Spectral Entropy (TF)', 'Channel 2 Spectral Kurtosis'
]

# Extract Y_true and drop NaN values
top_features_df = top_features_df.dropna(subset=['Y_true'])
X_full = top_features_df[top_12_features]
Y_true = top_features_df['Y_true']

# Initialize variables for best combination
best_combination = None
best_r2_score = float('-inf')
results = []
ypred_dict = {}

# Iterate over all combinations of 4 features
for combo in combinations(top_12_features, 4):
    X = X_full[list(combo)]
    model = LinearRegression()
    model.fit(X, Y_true)
    Y_pred = model.predict(X)
    r2 = r2_score(Y_true, Y_pred)
    results.append({'Feature Combination': combo, 'R² Score': r2})
    ypred_dict[str(combo)] = Y_pred
    if r2 > best_r2_score:
        best_r2_score = r2
        best_combination = combo
        best_Y_pred = Y_pred
        best_model = model

# Create a DataFrame for the best combination
best_results_df = X_full[list(best_combination)].copy()
best_results_df["Y_true"] = Y_true
best_results_df["Y_pred"] = best_Y_pred

# Generate regression equation
intercept = best_model.intercept_
coefficients = best_model.coef_
#equation = f"Y = {intercept:.4f} " + " + ".join([f"({coeff:.4f} * {feature})" for coeff, feature in zip(coefficients, best_combination)])
equation_terms = [f"{coeff:.4f} * {feature}" for coeff, feature in zip(coefficients, best_combination)]
equation = f"Y = {' + '.join(equation_terms)} + {intercept:.4f}"

# Define output file path
output_file = "Feature_Combinations_Results.xlsx"

# Save only the best combination input values and Y_pred to Excel
# Save results to Excel with an additional sheet for feature combinations
with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    features_df.to_excel(writer, sheet_name="Combined_Results", index=False)
    selected_features_df.to_excel(writer, sheet_name="Top_Features", index=False)
    pd.DataFrame(all_combinations, columns=["Feature 1", "Feature 2", "Feature 3", "Feature 4"]).to_excel(writer, sheet_name="Feature_Combinations", index=False)
    best_results_df.to_excel(writer, sheet_name="Best_Combination", index=False)

# Output the best combination and accuracy
print(f"Best 4-feature combination: {best_combination}")
print(f"Highest R² score: {best_r2_score}")
print(f"Results saved in {output_file} with all feature combinations listed in an additional sheet.")
print (equation)


Best 4-feature combination: ('Channel 2 RMS', 'Channel 3 PEAK', 'Channel 1 CREST FACTOR', 'Channel 3 Spectral Entropy')
Highest R² score: 0.9056157550679115
Results saved in Feature_Combinations_Results.xlsx with all feature combinations listed in an additional sheet.
Y = -81.1127 * Channel 2 RMS + 5.2743 * Channel 3 PEAK + -0.2935 * Channel 1 CREST FACTOR + -3.5409 * Channel 3 Spectral Entropy + 25.7351
